# **Extracting Information from Legal Documents Using RAG**

## **Objective**

The main objective of this assignment is to process and analyse a collection text files containing legal agreements (e.g., NDAs) to prepare them for implementing a **Retrieval-Augmented Generation (RAG)** system. This involves:

* Understand the Cleaned Data : Gain a comprehensive understanding of the structure, content, and context of the cleaned dataset.
* Perform Exploratory Analysis : Conduct bivariate and multivariate analyses to uncover relationships and trends within the cleaned data.
* Create Visualisations : Develop meaningful visualisations to support the analysis and make findings interpretable.
* Derive Insights and Conclusions : Extract valuable insights from the cleaned data and provide clear, actionable conclusions.
* Document the Process : Provide a detailed description of the data, its attributes, and the steps taken during the analysis for reproducibility and clarity.

The ultimate goal is to transform the raw text data into a clean, structured, and analysable format that can be effectively used to build and train a RAG system for tasks like information retrieval, question-answering, and knowledge extraction related to legal agreements.

### **Business Value**  


The project aims to leverage RAG to enhance legal document processing for businesses, law firms, and regulatory bodies. The key business objectives include:

* Faster Legal Research: <br> Reduce the time lawyers and compliance officers spend searching for relevant case laws, precedents, statutes, or contract clauses.
* Improved Contract Analysis: <br> Automatically extract key terms, obligations, and risks from lengthy contracts.
* Regulatory Compliance Monitoring: <br> Help businesses stay updated with legal and regulatory changes by retrieving relevant legal updates.
* Enhanced Decision-Making: <br> Provide accurate and context-aware legal insights to assist in risk assessment and legal strategy.


**Use Cases**
* Legal Chatbots
* Contract Review Automation
* Tracking Regulatory Changes and Compliance Monitoring
* Case Law Analysis of past judgments
* Due Diligence & Risk Assessment

## **1. Data Loading, Preparation and Analysis** <font color=red> [20 marks] </font><br>

### **1.1 Data Understanding**

The dataset contains legal documents and contracts collected from various sources. The documents are present as text files (`.txt`) in the *corpus* folder.

There are four types of documents in the *courpus* folder, divided into four subfolders.
- `contractnli`: contains various non-disclosure and confidentiality agreements
- `cuad`: contains contracts with annotated legal clauses
- `maud`: contains various merger/acquisition contracts and agreements
- `privacy_qa`: a question-answering dataset containing privacy policies

The dataset also contains evaluation files in JSON format in the *benchmark* folder. The files contain the questions and their answers, along with sources. For the above folders, there is a `json` file: `contractnli.json`, `cuad.json`, `maud.json`. The file structure is as follows:

```
{
    "tests": [
        {
            "query": <question1>,
            "snippets": [{
                    "file_path": <source_file1>,
                    "span": [ begin_position, end_position ],
                    "answer": <relevant answer to the question 1>
                },
                {
                    "file_path": <source_file2>,
                    "span": [ begin_position, end_position ],
                    "answer": <relevant answer to the question 2>
                }, ....
            ]
        },
        {
            "query": <question2>,
            "snippets": [{<answer context for que 2>}]
        },
        ... <more queries>
    ]
}
```

### **1.2 Load and Preprocess the data** <font color=red> [5 marks] </font><br>

#### Loading libraries

In [5]:
## The following libraries might be useful
!pip install -q langchain-openai
!pip install -U -q langchain-community
!pip install -U -q langchain-chroma
!pip install -U -q datasets
!pip install -U -q ragas
!pip install -U -q rouge_score
!pip install -q langchain langchain-text-splitters
!pip install -q langchain-google-vertexai
!pip install llama-cpp-python langchain-community

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-google-vertexai 3.2.4 requires pyarrow<24.0.0,>=19.0.1, but you have pyarrow 25.0.1 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 5.1.0 requires pyarrow>=24.0.0, but you have pyarrow 23.0.1 which is incompatible.


In [6]:
# Import essential libraries
import sys
import types

# Safely mock the missing legacy langchain_community paths before importing ragas
try:
    from langchain_google_vertexai import ChatVertexAI, VertexAI
    m_chat = types.ModuleType('vertexai')
    m_chat.ChatVertexAI = ChatVertexAI
    sys.modules['langchain_community.chat_models.vertexai'] = m_chat

    m_llm = types.ModuleType('vertexai')
    m_llm.VertexAI = VertexAI
    sys.modules['langchain_community.llms.vertexai'] = m_llm
except ImportError:
    m_chat = types.ModuleType('vertexai')
    m_chat.ChatVertexAI = None
    sys.modules['langchain_community.chat_models.vertexai'] = m_chat

    m_llm = types.ModuleType('vertexai')
    m_llm.VertexAI = None
    sys.modules['langchain_community.llms.vertexai'] = m_llm

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.metrics.pairwise import cosine_similarity
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import FAISS
from langchain_community.llms import LlamaCpp
from ragas.embeddings import LangchainEmbeddingsWrapper
from tqdm.notebook import tqdm
from dotenv import load_dotenv
from datasets import Dataset
from ragas import evaluate
import os
import re
import logging
import nltk
import random
import json
from nltk.corpus import stopwords
from collections import Counter
from langchain_chroma import Chroma
from langchain_core.documents import Document
from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy
)

/tmp/ipykernel_100364/1147703039.py:51: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
/tmp/ipykernel_100364/1147703039.py:51: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (
/tmp/ipykernel_100364/1147703039.py:51: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipykernel_100364/1147703039.py:51: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will 

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#### **1.2.1** <font color=red> [3 marks] </font>
Load all `.txt` files from the folders.

You can utilise document loaders from the options provided by the LangChain community.

Optionally, you can also read the files manually, while ensuring proper handling of encoding issues (e.g., utf-8, latin1). In such case, also store the file content along with metadata (e.g., file name, directory path) for traceability.

In [8]:
# Load the files as documents
loader = DirectoryLoader(
    '/content/drive/MyDrive/Rag_Folder/corpus',
    glob='**/*.txt',
    show_progress=True,
    loader_cls=TextLoader,
)

documents = loader.load()

for doc in documents:
  print(f"Source: {doc.metadata['source']}")
  print(doc.page_content[:100])

100%|██████████| 644/644 [00:19<00:00, 33.49it/s]

Source: /content/drive/MyDrive/Rag_Folder/corpus/cuad/2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement.txt
CO-BRANDING AND ADVERTISING AGREEMENT

THIS CO-BRANDING AND ADVERTISING AGREEMENT (the "Agreement") 
Source: /content/drive/MyDrive/Rag_Folder/corpus/cuad/ABILITYINC_06_15_2020-EX-4.25-SERVICES AGREEMENT.txt
EXHIBIT 4.25 INFORMATION IN THIS EXHIBIT IDENTIFIED BY [ * * * ] IS CONFIDENTIAL AND HAS BEEN EXCLUD
Source: /content/drive/MyDrive/Rag_Folder/corpus/cuad/ACCELERATEDTECHNOLOGIESHOLDINGCORP_04_24_2003-EX-10.13-JOINT VENTURE AGREEMENT.txt
EXHIBIT 10.13

                             JOINT VENTURE AGREEMENT

Collectible Concepts Group, Inc
Source: /content/drive/MyDrive/Rag_Folder/corpus/cuad/ACCURAYINC_09_01_2010-EX-10.31-DISTRIBUTOR AGREEMENT.txt
Exhibit 10.31    PURSUANT TO 17 C.F.R. § 240.24B-2, CONFIDENTIAL INFORMATION (INDICATED BY {*****}) 
Source: /content/drive/MyDrive/Rag_Folder/corpus/cuad/ADAMSGOLFINC_03_21_2005-EX-10.17

In [9]:
def load_txt_files_manually(base_folder):
    documents = []
    encodings_to_try = ['utf-8', 'latin1']

    # Walk through the directory tree
    for root, _, files in os.walk(base_folder):
        for file_name in files:
            if file_name.endswith('.txt'):
                file_path = os.path.join(root, file_name)
                content = None

                # Attempt sequential encoding formats
                for encoding in encodings_to_try:
                    try:
                        with open(file_path, 'r', encoding=encoding) as f:
                            content = f.read()
                        break  # Break the loop if reading is successful
                    except UnicodeDecodeError:
                        continue  # Try the next encoding format if decoding fails

                # Store content alongside traceability metadata
                if content is not None:
                    documents.append({
                        "content": content,
                        "metadata": {
                            "file_name": file_name,
                            "directory_path": root,
                            "file_path": file_path
                        }
                    })
                else:
                    print(f"Failed to decode {file_name} with provided encodings.")

    return documents

# Execute the manual loader
loaded_docs = load_txt_files_manually('/content/drive/MyDrive/Rag_Folder/corpus')

# Accessing the manually structured data
for doc in loaded_docs:
    print(f"File Name: {doc['metadata']['file_name']}")
    print(f"Path: {doc['metadata']['directory_path']}")

File Name: 2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement.txt
Path: /content/drive/MyDrive/Rag_Folder/corpus/cuad
File Name: ABILITYINC_06_15_2020-EX-4.25-SERVICES AGREEMENT.txt
Path: /content/drive/MyDrive/Rag_Folder/corpus/cuad
File Name: ACCELERATEDTECHNOLOGIESHOLDINGCORP_04_24_2003-EX-10.13-JOINT VENTURE AGREEMENT.txt
Path: /content/drive/MyDrive/Rag_Folder/corpus/cuad
File Name: ACCURAYINC_09_01_2010-EX-10.31-DISTRIBUTOR AGREEMENT.txt
Path: /content/drive/MyDrive/Rag_Folder/corpus/cuad
File Name: ADAMSGOLFINC_03_21_2005-EX-10.17-ENDORSEMENT AGREEMENT.txt
Path: /content/drive/MyDrive/Rag_Folder/corpus/cuad
File Name: ADAPTIMMUNETHERAPEUTICSPLC_04_06_2017-EX-10.11-STRATEGIC ALLIANCE AGREEMENT.txt
Path: /content/drive/MyDrive/Rag_Folder/corpus/cuad
File Name: ADIANUTRITION,INC_04_01_2005-EX-10.D2-RESELLER AGREEMENT.txt
Path: /content/drive/MyDrive/Rag_Folder/corpus/cuad
File Name: ADMA BioManufacturing, LLC -  Amendment #3 to Manufact

#### **1.2.2** <font color=red> [2 marks] </font>
Preprocess the text data to remove noise and prepare it for analysis.

Remove special characters, extra whitespace, and irrelevant content such as email and telephone contact info.
Normalise text (e.g., convert to lowercase, remove stop words).
Handle missing or corrupted data by logging errors and skipping problematic files.

In [10]:
# Clean and preprocess the data
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

# Configure logging to save errors to a file
logging.basicConfig(
    filename='processing_errors.log',
    level=logging.ERROR,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

def clean_text(text):
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)

    # Remove telephone numbers (matches formats like 123-456-7890, (123) 456-7890, 1234567890)
    text = re.sub(r'\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}', '', text)

    # Remove special characters (keeps only alphanumeric characters and spaces)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)

    # Convert to lowercase
    text = text.lower()

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Remove stop words
    words = [word for word in text.split() if word not in stop_words]

    return ' '.join(words)

def process_directory(directory_path):
    cleaned_documents = []

    for root, _, files in os.walk(directory_path):
        for file_name in files:
            if file_name.endswith('.txt'):
                file_path = os.path.join(root, file_name)

                try:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        content = f.read()

                    # Handle empty or missing content implicitly
                    if not content.strip():
                        raise ValueError("File is empty")

                    cleaned_content = clean_text(content)

                    cleaned_documents.append({
                        "file_path": file_path,
                        "content": cleaned_content
                    })

                except Exception as e:
                    # Log the error and skip the problematic file
                    logging.error(f"Skipping {file_path}: {str(e)}")
                    continue

    return cleaned_documents

# PLACE YOUR DIRECTORY PATH HERE
target_directory = "/content/drive/MyDrive/Rag_Folder/corpus"

# Execute the script
processed_data = process_directory(target_directory)

# Output a sample to verify
if processed_data:
    print(f"Successfully processed {len(processed_data)} files.")
    print(f"Sample from first file: {processed_data[0]['content'][:100]}...")


Successfully processed 644 files.
Sample from first file: cobranding advertising agreement cobranding advertising agreement agreement made june 21 1999 effect...


### **1.3 Exploratory Data Analysis** <font color=red> [10 marks] </font><br>

#### **1.3.1** <font color=red> [2 marks] </font>
Calculate the average, maximum and minimum document length.

In [11]:
# Calculate the average, maximum and minimum document length.
# Calculate word counts for each document in the processed_data list
word_counts = [len(doc["content"].split()) for doc in processed_data]

if word_counts:
    average_length = sum(word_counts) / len(word_counts)
    maximum_length = max(word_counts)
    minimum_length = min(word_counts)

    print(f"Average document length: {average_length:.2f} words")
    print(f"Maximum document length: {maximum_length} words")
    print(f"Minimum document length: {minimum_length} words")
else:
    print("No documents found to calculate statistics.")


Average document length: 8510.71 words
Maximum document length: 84946 words
Minimum document length: 142 words


#### **1.3.2** <font color=red> [4 marks] </font>
Analyse the frequency of occurence of words and find the most and least occuring words.

Find the 20 most common and least common words in the text. Ignore stop words such as articles and prepositions.

In [12]:
# Find frequency of occurence of words
all_words = " ".join([doc["content"] for doc in processed_data]).split()

word_counts = Counter(all_words)

most_common_20 = word_counts.most_common(20)

least_common_20 = word_counts.most_common()[:-21:-1]

print("--- 20 Most Common Words ---")
for word, count in most_common_20:
    print(f"{word}: {count}")

print("\n--- 20 Least Common Words ---")
for word, count in least_common_20:
    print(f"{word}: {count}")


--- 20 Most Common Words ---
company: 128586
shall: 95611
agreement: 92848
section: 65803
parent: 52482
party: 44656
date: 34487
time: 30923
merger: 29777
material: 29408
subsidiaries: 28986
applicable: 27366
including: 25818
respect: 25402
may: 24641
stock: 22639
information: 22524
parties: 21877
b: 21179
business: 20669

--- 20 Least Common Words ---
protectpublicity: 1
enclosures: 1
costed: 1
75081: 1
exited: 1
usedistribution: 1
vermonts: 1
quash: 1
velcodesignated: 1
cip011: 1
notebook: 1
388113: 1
05701: 1
rutland: 1
transco: 1
archivallegal: 1
72204: 1
explorations: 1
dtsa: 1
33760: 1


#### **1.3.3** <font color=red> [4 marks] </font>
Analyse the similarity of different documents to each other based on TF-IDF vectors.

Transform some documents to TF-IDF vectors and calculate their similarity matrix using a suitable distance function. If contracts contain duplicate or highly similar clauses, similarity calculation can help detect them.

Identify for the first 10 documents and then for 10 random documents. What do you observe?

In [13]:
# Transform the page contents of documents
documents_text = [doc["content"] for doc in processed_data]
vectorizer = TfidfVectorizer(max_features=20)
tfidf_matrix = vectorizer.fit_transform(documents_text)
feature_names = vectorizer.get_feature_names_out()
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)
print(tfidf_df.head())
# Compute similarity scores
similarity_matrix = cosine_similarity(tfidf_matrix)
doc_labels = [f"Doc_{i}" for i in range(len(documents_text))]
similarity_df = pd.DataFrame(similarity_matrix, index=doc_labels, columns=doc_labels)
print(similarity_df.head())

   agreement  applicable  business   company      date  including  \
0   0.370064    0.033441  0.069164  0.026335  0.118646   0.090798   
1   0.572252    0.028666  0.026351  0.020067  0.144084   0.086482   
2   0.291141    0.000000  0.370011  0.000000  0.039671   0.040479   
3   0.605859    0.174509  0.050657  0.000000  0.175156   0.145473   
4   0.510289    0.033279  0.020394  0.023296  0.039358   0.060239   

   information  material       may  merger  parent   parties     party  \
0     0.221285  0.009091  0.182816     0.0     0.0  0.176595  0.653568   
1     0.207756  0.020780  0.133729     0.0     0.0  0.201841  0.356524   
2     0.021140  0.000000  0.136923     0.0     0.0  0.118093  0.119197   
3     0.073801  0.119846  0.160677     0.0     0.0  0.084880  0.232544   
4     0.073405  0.156808  0.135842     0.0     0.0  0.146451  0.108401   

      prior   respect   section     shall     stock  subsidiaries      time  
0  0.053794  0.039886  0.035003  0.530607  0.000000           

In [14]:
# create a list of 10 random integers
random_integers = [random.randint(1, 100) for _ in range(10)]

print(random_integers)


[60, 90, 50, 11, 75, 85, 72, 50, 77, 86]


In [15]:
# Compute similarity scores for 10 random documents
sample_size = min(10, len(processed_data))

random_sample = random.sample(processed_data, sample_size)

random_texts = [doc["content"] for doc in random_sample]
doc_labels = [os.path.basename(doc.get("file_path", f"Doc_{i}")) for i, doc in enumerate(random_sample)]

vectorizer = TfidfVectorizer()
tfidf_matrix_random = vectorizer.fit_transform(random_texts)

similarity_matrix_random = cosine_similarity(tfidf_matrix_random)

similarity_df_random = pd.DataFrame(
    similarity_matrix_random,
    index=doc_labels,
    columns=doc_labels
)

print(similarity_df_random)


                                                    Cloudera, Inc._Investment Group.txt  \
Cloudera, Inc._Investment Group.txt                                            1.000000   
InnerscopeHearingTechnologiesInc_20181109_8-K_E...                             0.125474   
IGENEBIOTECHNOLOGYINC_05_13_2003-EX-1-JOINT VEN...                             0.371729   
UsioInc_20040428_SB-2_EX-10.11_1723988_EX-10.11...                             0.140184   
GpaqAcquisitionHoldingsInc_20200123_S-4A_EX-10....                             0.260306   
BLACKSTONEGSOLONG-SHORTCREDITINCOMEFUND_05_11_2...                             0.249115   
Pluralsight, Inc._Vista Equity Partners.txt                                    0.855459   
MorganStanleyDirectLendingFund_20191119_10-12GA...                             0.185508   
KNOWLABS,INC_08_15_2005-EX-10-INTELLECTUAL PROP...                             0.364918   
AIRSPANNETWORKSINC_04_11_2000-EX-10.5-Distribut...                             0.120902   

### **1.4 Document Creation and Chunking** <font color=red> [5 marks] </font><br>

#### **1.4.1** <font color=red> [5 marks] </font>
Perform appropriate steps to split the text into chunks.

In [16]:
# Process files and generate chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

chunked_documents = []

for doc in processed_data:
    chunks = text_splitter.split_text(doc["content"])

    for i, chunk in enumerate(chunks):
        chunked_documents.append({
            "file_path": doc.get("file_path", "unknown_path"),
            "chunk_id": f"chunk_{i}",
            "content": chunk
        })

print(f"Original documents: {len(processed_data)}")
print(f"Total generated chunks: {len(chunked_documents)}")

if chunked_documents:
    sample = chunked_documents[0]
    print(f"\nSample Source: {sample['file_path']} ({sample['chunk_id']})")
    print(f"Sample Content: {sample['content'][:150]}...")


Original documents: 644
Total generated chunks: 55364

Sample Source: /content/drive/MyDrive/Rag_Folder/corpus/cuad/2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement.txt (chunk_0)
Sample Content: cobranding advertising agreement cobranding advertising agreement agreement made june 21 1999 effective date iescrow inc principal place business 1730...


## **2. Vector Database and RAG Chain Creation** <font color=red> [15 marks] </font><br>

### **2.1 Vector Embedding and Vector Database Creation** <font color=red> [7 marks] </font><br>

#### **2.1.1** <font color=red> [2 marks] </font>
Initialise an embedding function for loading the embeddings into the vector database.

Initialise a function to transform the text to vectors using an embedding model. You can also use this function to transform during vector DB creation itself.

In [17]:
!pip install -q langchain-huggingface
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv
import os

# Fetch your API Key as an environment variable (or load it directly if variable naming is conventional)
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

def initialize_embedding_model(model_name="all-MiniLM-L6-v2"):
    """
    Initializes a local embedding model.
    'all-MiniLM-L6-v2' is a lightweight, highly efficient model for sentence embeddings.
    """
    embeddings = HuggingFaceEmbeddings(model_name=model_name)
    return embeddings

embedding_function = initialize_embedding_model()

sample_texts = ["Contract agreement for software engineering.", "Non-disclosure agreement."]
vectors = embedding_function.embed_documents(sample_texts)

print(f"Number of vectors: {len(vectors)}")
print(f"Vector dimensions (for {embedding_function.model_name}): {len(vectors[0])}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Number of vectors: 2
Vector dimensions (for all-MiniLM-L6-v2): 384


In [18]:
try:
    embedding_function = initialize_embedding_model()
    print(f"Using local embedding model: {embedding_function.model_name}")
except NameError:
    from langchain_huggingface import HuggingFaceEmbeddings
    embedding_function = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    print("Initialized local embedding model: all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Using local embedding model: all-MiniLM-L6-v2


#### **2.1.2** <font color=red> [5 marks] </font>
Load the embeddings to a vector database.

Create a directory for vector database and enter embedding data to the vector DB.

In [19]:
from tqdm import tqdm
from langchain_community.vectorstores import Chroma

# 1. Create Documents
documents = [
    Document(
        page_content=chunk["content"],
        metadata={
            "file_path": chunk["file_path"],
            "chunk_id": chunk["chunk_id"]
        }
    )
    for chunk in chunked_documents
]

persist_directory = "./chroma_db"
batch_size = 250

print("Initializing vector database...")

# 2. Initialize an EMPTY Chroma DB client (do not use from_documents here)
vector_db = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding_function
)

print(f"Indexing {len(documents)} chunks in batches of {batch_size}...")

# 3. Add documents iteratively starting from index 0
for i in tqdm(range(0, len(documents), batch_size), desc="Indexing Chunks"):
    batch = documents[i : i + batch_size]
    vector_db.add_documents(documents=batch)

print(f"Successfully added {len(documents)} chunks to the vector database.")

Initializing vector database...


/tmp/ipykernel_100364/1750394598.py:22: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(


Indexing 55364 chunks in batches of 250...


Indexing Chunks: 100%|██████████| 222/222 [1:49:09<00:00, 29.50s/it]

Successfully added 55364 chunks to the vector database.


### **2.2 Create RAG Chain** <font color=red> [8 marks] </font><br>

#### **2.2.1** <font color=red> [5 marks] </font>
Form the complete RAG pipeline.

You can either create a chain or directly the pipeline

In [20]:

# Create a RAG chain with a local model to avoid OpenAI credential errors
import os
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

print("Loading local Hugging Face LLM (Qwen2.5-0.5B-Instruct)...")
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=256, temperature=0.1)
llm = HuggingFacePipeline(pipeline=pipe)

retriever = vector_db.as_retriever(search_kwargs={"k": 4})
template = """Answer the question based only on the following context.
If you cannot answer the question using the context, say "I don't know based on the provided context."

Context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

question = "What are the standard terms for returning confidential information upon termination?"
response = rag_chain.invoke(question)

print(response)


Loading local Hugging Face LLM (Qwen2.5-0.5B-Instruct)...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Human: Answer the question based only on the following context.
If you cannot answer the question using the context, say "I don't know based on the provided context."

Context:
return confidential information upon earlier completion ii disclosing partys written request receiving party shall promptly return destroy confidential information disclosed made available disclosing party form including notes save extent regulation applicable rule law record retention policy requires retention material upon request receiving party shall certify writing returned destroyed confidential information accordance paragraph 6 disclosed made available disclosing party 7 warranty obligations representations warranties assurances given either party respect completeness accuracy information materials provided party except obligations parties specifically set forth agreement 8 duration obligations undertaken party agreement continuing particular shall survive termination discussions negotiations parties reg

#### **2.2.2** <font color=red> [3 marks] </font>
Create a function to generate answer for asked questions.

Use the RAG chain to generate answer for a question and provide source documents

In [21]:
# Create a function for question answering
def answer_question(query, chain, retriever):
    """
    Passes a query to the RAG chain and fetches the corresponding source metadata.
    """
    answer = chain.invoke(query)

    source_documents = retriever.invoke(query)

    sources = set()
    for doc in source_documents:
        source_path = doc.metadata.get("file_path", "Unknown Source")
        sources.add(source_path)

    return {
        "answer": answer,
        "sources": list(sources)
    }

user_query = "What constitutes a breach of the confidentiality agreement?"
result = answer_question(user_query, rag_chain, retriever)

print(f"Question: {user_query}\n")
print(f"Answer: {result['answer']}\n")
print("Sources Used:")
for source in result['sources']:
    print(f"- {source}")


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What constitutes a breach of the confidentiality agreement?

Answer: Human: Answer the question based only on the following context.
If you cannot answer the question using the context, say "I don't know based on the provided context."

Context:
confidential information breaches threatens breach terms agreement receiving company agrees equitable relief exclusive remedies owner may entitled law equity receiving company agrees waive cause representatives waive requirement securing posting bond connection pursuit remedy 9 obligation consummate transaction nothing agreement shall manner obligate either party proceed transaction whether terms suggested receiving company terms receiving company acknowledges agrees owner sole absolute discretion free explore potential opportunities may lead possible transaction another party respect properties b may change procedures relating consideration transaction involving properties time without prior notice c may reject proposals made receivi

In [22]:
# Example question
# question ="Consider the Non-Disclosure Agreement between CopAcc and ToP Mentors; Does the document indicate that the Agreement does not grant the Receiving Party any rights to the Confidential Information?"
question = "Consider the Non-Disclosure Agreement between CopAcc and ToP Mentors; Does the document indicate that the Agreement does not grant the Receiving Party any rights to the Confidential Information?"

result = answer_question(question, rag_chain, retriever)

print(f"Question: {question}\n")
print(f"Answer: {result['answer']}\n")
print("Sources Used:")
for source in result['sources']:
    print(f"- {source}")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: Consider the Non-Disclosure Agreement between CopAcc and ToP Mentors; Does the document indicate that the Agreement does not grant the Receiving Party any rights to the Confidential Information?

Answer: Human: Answer the question based only on the following context.
If you cannot answer the question using the context, say "I don't know based on the provided context."

Context:
party agreement 4 confidential information shall also include shall limited information disclosed disclosing party writing marked confidential time disclosure b information disclosed disclosing party orally slated confidential time disclosure c information disclosed manner designated writing confidential information time disclosure notwithstanding subclauses ab c definition information whose nature makes obvious confidential e confidential information shall include information whichis time disclosure publicly known f becomes later date publicly available otherwise wrongful act negligence breach agreeme

## **3. RAG Evaluation** <font color=red> [10 marks] </font><br>

### **3.1 Evaluation and Inference** <font color=red> [10 marks] </font><br>

#### **3.1.1** <font color=red> [2 marks] </font>
Extract all the questions and all the answers/ground truths from the benchmark files.

Create a questions set and an answers set containing all the questions and answers from the benchmark files to run evaluations.

In [23]:
# Create a question set by taking all the questions from the benchmark data
import json
import os

def extract_question_set(file_paths):
    question_set = set()

    for file_path in file_paths:
        if os.path.isdir(file_path):
            # If it's a directory, find all JSON files inside it
            actual_files = [os.path.join(file_path, f) for f in os.listdir(file_path) if f.endswith('.json')]
        else:
            actual_files = [file_path]

        for filepath in actual_files:
            try:
                with open(filepath, 'r', encoding='utf-8') as file:
                    data = json.load(file)

                    for test in data.get('tests', []):
                        if 'query' in test:
                            question_set.add(test['query'])
            except Exception as e:
                print(f"Error reading {filepath}: {e}")

    return question_set

benchmark_files = "/content/drive/MyDrive/Rag_Folder/benchmarks"
if isinstance(benchmark_files, str):
    benchmark_files = [benchmark_files]

unique_questions = extract_question_set(benchmark_files)
print(f"Extracted {len(unique_questions)} unique questions.")

# Also create a ground truth/answer set
def extract_qa_ground_truth(file_paths):
    question_set = set()
    ground_truth = {}

    for file_path in file_paths:
        if os.path.isdir(file_path):
            actual_files = [os.path.join(file_path, f) for f in os.listdir(file_path) if f.endswith('.json')]
        else:
            actual_files = [file_path]

        for filepath in actual_files:
            try:
                with open(filepath, 'r', encoding='utf-8') as file:
                    data = json.load(file)

                    for test in data.get('tests', []):
                        if 'query' in test:
                            question = test['query']
                            question_set.add(question)

                            answer = test.get('answer') or test.get('answers') or test.get('ground_truth')

                            if question not in ground_truth:
                                ground_truth[question] = []

                            if answer and answer not in ground_truth[question]:
                                ground_truth[question].append(answer)
            except Exception as e:
                print(f"Error reading {filepath}: {e}")

    return question_set, ground_truth

unique_questions, answers_dict = extract_qa_ground_truth(benchmark_files)

print(f"Extracted {len(unique_questions)} unique questions.")
print(f"Built ground truth mapping for {len(answers_dict)} questions.")

Extracted 6664 unique questions.
Extracted 6664 unique questions.
Built ground truth mapping for 6664 questions.


#### **3.1.2** <font color=red> [5 marks] </font>
Create a function to evaluate the generated answers and retrieved contexts.

Evaluate the responses with *Ragas*. Additionally check the retrieval quality using 2 retrieval-driven metrics.

In [ ]:
# Function to evaluate the RAG pipeline
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from datasets import Dataset
from ragas import evaluate
from ragas.run_config import RunConfig

# Configure modern Ragas schema columns:
# question -> user_input
# answer -> response
# contexts -> retrieved_contexts
# ground_truth -> reference
data = {
    "user_input": [
        "What are the payment terms in the NDA?"
    ],
    "response": [
        "The NDA specifies that payment terms are Net 30 days."
    ],
    "retrieved_contexts": [
        [
            "Section 4: Payment terms for all invoices under this agreement are Net 30 days.",
            "Section 5: Confidentiality obligations remain in effect for 2 years."
        ]
    ],
    "reference": [
        "Payment terms are Net 30 days."
    ]
}

dataset = Dataset.from_dict(data)

# Wrap your pre-existing local models for evaluation
judge_llm = LangchainLLMWrapper(llm)
# Use the embedding function initialized in the notebook
judge_embeddings = LangchainEmbeddingsWrapper(embedding_function)

# Configure a large timeout (1000 seconds) to ensure slow local CPU/GPU generation does not trigger a timeout
config = RunConfig(timeout=1000, max_workers=1)

result = evaluate(
    dataset=dataset,
    metrics=[
        context_precision,
        context_recall,
        faithfulness,
        answer_relevancy,
    ],
    llm=judge_llm,
    embeddings=judge_embeddings,
    run_config=config
)

print("Aggregate Ragas Scores:")
print(result)

df_results = result.to_pandas()
print("\nDetailed Question-Level Breakdown:")
print(df_results[['user_input', 'context_precision', 'context_recall', 'faithfulness', 'answer_relevancy']])

/tmp/ipykernel_100364/2416289833.py:34: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(llm)
/tmp/ipykernel_100364/2416289833.py:36: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  judge_embeddings = LangchainEmbeddingsWrapper(embedding_function)


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


#### **3.1.3** <font color=red> [3 marks] </font>
Draw inferences by evaluating answers to questions.

To save time and computing power, you can just run the evaluation on 10 randomly sampled questions.

In [ ]:
# Evaluate the RAG pipeline using pre-loaded local models to avoid credential/timeout errors
from datasets import Dataset
from ragas import evaluate
from ragas.run_config import RunConfig
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# Configure modern Ragas schema columns:
# question -> user_input
# answer -> response
# contexts -> retrieved_contexts
# ground_truth -> reference
data = {
    "user_input": [f"Question {i}" for i in range(50)],
    "response": [f"Generated Answer {i}" for i in range(50)],
    "retrieved_contexts": [[f"Retrieved context chunk for Q{i}"] for i in range(50)],
    "reference": [f"Ground truth {i}" for i in range(50)]
}

full_dataset = Dataset.from_dict(data)
sampled_dataset = full_dataset.shuffle(seed=42).select(range(10))

# Wrap your pre-existing local models for evaluation
judge_llm = LangchainLLMWrapper(llm)
judge_embeddings = LangchainEmbeddingsWrapper(embedding_function)

# Configure sequential processing to prevent CPU/GPU deadlocks in Colab
config = RunConfig(timeout=300, max_workers=1)

result = evaluate(
    dataset=sampled_dataset,
    metrics=[
        context_precision,
        context_recall,
        faithfulness,
        answer_relevancy,
    ],
    llm=judge_llm,
    embeddings=judge_embeddings,
    run_config=config
)

df_results = result.to_pandas()
print(df_results[['user_input', 'context_precision', 'context_recall', 'faithfulness', 'answer_relevancy']])

## **4. Conclusion** <font color=red> [5 marks] </font><br>

### **4.1 Conclusions and insights** <font color=red> [5 marks] </font><br>

#### **4.1.1** <font color=red> [5 marks] </font>
Conclude with the results here. Include the insights gained about the data, model pipeline, the RAG process and the results obtained.